# Figures for Silent Alignment

This notebook draws the seven figures of the paper from the simulation archives in `data/`, together with the high-dimensional predictions, and saves them to `figures/pdf/`.

In [ ]:
from pathlib import Path
import json
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root (the folder containing src/).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import DATA_DIR, FIGURES_DIR
from src.theory import SignalLaw, make_theory


def paper_density(x):
    """Same smooth non-uniform law as src.laws.paper_phi."""
    x = np.asarray(x, dtype=float)
    a, b = 0.35, 2.0
    u = (x - a) / (b - a)
    left_bump = 0.75 * np.exp(-0.5 * ((u - 0.30) / 0.13) ** 2)
    right_bump = 0.45 * np.exp(-0.5 * ((u - 0.70) / 0.18) ** 2)
    shoulder = 0.16 * (1.0 + 0.35 * np.cos(2.0 * np.pi * u))
    out = np.zeros_like(x)
    mask = (x >= a) & (x <= b)
    out[mask] = np.maximum(left_bump[mask] + right_bump[mask] + shoulder[mask], 0.0)
    return out


law = SignalLaw.from_pdf(paper_density, (0.35, 2.0), name="paper_two_bump")

x_grid = np.linspace(0.01, 3.0, 500)

def load_result_npz(path):
    """Load a simulation archive (NumPy only)."""
    with np.load(path, allow_pickle=False) as data:
        out = {key: data[key] for key in data.files if not key.endswith("_json")}
        out["config"] = json.loads(str(data["config_json"]))
        out["dimensions"] = json.loads(str(data["dimensions_json"]))
    return out

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "U": "#2B6F9E",
    "V": "#009E8E",
    "Z": "#6D4C9B",
    "C": "#202124",
    "accent": "#D95F02",
    "rho1": "#D1495B",
    "rho2": "#F28E2B",
    "rho3": "#1B9E77",
    "rho4": "#4E79A7",
    "rho5": "#B07AA1",
    "muted": "#7A7F87",
}

mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.family": "serif",
    "axes.unicode_minus": False,
    "figure.dpi": 120,
    "savefig.dpi": 240,
    "font.size": 11,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 13,
    "figure.titlesize": 15,
    "axes.grid": True,
    "grid.alpha": 0.2,
})


figwidth = 10


In [ ]:
FINITE_PATHS = {
    "fig2": DATA_DIR / "figure02_visual_abstract.npz",
    "fig3_under": DATA_DIR / "figure03_convergence_under.npz",
    "fig3_over": DATA_DIR / "figure03_convergence_over.npz",
    "fig4": DATA_DIR / "figure04_density_comparison.npz",
    "fig5_under": DATA_DIR / "figure05_under.npz",
    "fig5_over_l0": DATA_DIR / "figure05_over_exponent_l0.npz",
    "fig5_over_l1": DATA_DIR / "figure05_over_exponent_l1.npz",
    "fig6_continuous": DATA_DIR / "figure06_continuous_rates.npz",
    "fig6_gapped": DATA_DIR / "figure06_gapped_rates.npz",
    "fig7_rho": DATA_DIR / "figure07_rho_staircase.npz",
    "fig7_large": DATA_DIR / "figure07_large_dynamics.npz",
    "fig8": DATA_DIR / "figure08_spectral_measures.npz",
}


class LazySimulation:
    def __init__(self, path, label):
        self.path = Path(path)
        self.label = label
        self._data = None

    def load(self):
        if self._data is None:
            print(f"loading {self.label}: {self.path}")
            self._data = load_result_npz(self.path)
        return self._data

    def __getitem__(self, key):
        return self.load()[key]

    def __contains__(self, key):
        return key in self.load()


def maybe_load(path, label):
    path = Path(path)
    if path.exists():
        print(f"registered {label}: {path}")
        return LazySimulation(path, label)
    print(f"missing {label}: {path}")
    return None


def darken_color(color, amount=0.15):
    import matplotlib.colors as mcolors

    r, g, b = mcolors.to_rgb(color)
    factor = 1.0 - amount
    return (factor * r, factor * g, factor * b)


finite = {name: maybe_load(path, f"finite {name}") for name, path in FINITE_PATHS.items()}


In [ ]:
def arr(data, key):
    if data is None:
        return None
    if isinstance(data, LazySimulation):
        data = data.load()
    return data[key]


def loaded_dict(data):
    if data is None:
        return None
    return data.load() if isinstance(data, LazySimulation) else data


def has(data, key):
    return data is not None and key in data


def savefig(fig, name):
    path = FIGURES_DIR / name
    fig.savefig(path)
    print(path)


def rho_color(index):
    return [COLORS["rho1"], COLORS["rho2"], COLORS["rho3"], COLORS["rho4"], COLORS["rho5"]][index % 5]


def finite_times(data):
    return np.asarray(arr(data, "times"), dtype=float)


def finite_mean(data, key, rho_index=0):
    value = np.asarray(arr(data, key), dtype=float)
    if value.ndim < 3:
        return value
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(value[rho_index], axis=-1)


def regular_values(values, *, remove_zero=True, zero_tol=1e-10):
    vals = np.asarray(values, dtype=float).reshape(-1)
    vals = vals[np.isfinite(vals)]
    if remove_zero:
        vals = vals[np.abs(vals) > zero_tol]
    return vals


def normalize_curve(y):
    y = np.asarray(y, dtype=float)
    scale = np.nanmax(y)
    return y / scale if np.isfinite(scale) and scale > 0 else y


def theory_from_config(data):
    """High-dimensional theory with the parameters stored in an archive."""
    cfg = loaded_dict(data)["config"]
    return make_theory(
        law,
        kappa=float(cfg["kappa"]),
        tau=float(cfg["tau"]),
        kappa_star=float(cfg["kappa_star"]),
        rho=float(np.asarray(cfg["rhos"], dtype=float).reshape(-1)[0]),
        n_quad=600,
    )


def edge_power_pdf(ell, a=0.35, b=2.0):
    norm = (ell + 1.0) / ((b - a) ** (ell + 1.0))
    def pdf(x):
        x = np.asarray(x, dtype=float)
        out = np.zeros_like(x)
        mask = (x >= a) & (x <= b)
        out[mask] = norm * (x[mask] - a) ** ell
        return out
    return pdf


## Figure 2: Visual abstract

Finite simulations against the high-dimensional formulas, in the overparameterized regime. Left: alignment error on the timescale $t = \Theta(1)$. Right: loss against $s = t/\log(1/\rho)$, with histograms of the singular values of $Z$ at three values of $s$.

In [ ]:
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.transforms import Bbox
    
data = finite["fig2"]

times = finite_times(data)
mask = times > 0.0
rhos = np.asarray(arr(data, "rhos"))
theory = theory_from_config(data)
proj0 = 2 * (1 - theory.kappa) * min(theory.kappa, theory.beta * theory.kappa_star)

ALIGN_COLOR = "#2F4858"
LOSS_COLOR = "#8C510A"
curve_lw = 1.7
faint = dict(lw=1.3, ls="--", alpha=0.4, zorder=1)
limit_style = dict(color="black", lw=curve_lw, alpha=0.75, zorder=3)


def faint_arrow(ax, start, end, color):
    """Faint arrow across the dashed curves, pointing towards smaller rho, labelled above its middle."""
    ax.annotate("", xy=end, xytext=start, zorder=5,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.1, alpha=0.5, mutation_scale=10))
    ax.text(0.5 * (start[0] + end[0]), start[1] + 0.025, r"$\rho \to 0$", color=color, alpha=0.7,
            ha="center", va="bottom", fontsize=11, zorder=5)


def quantity_legend(ax, solid, dashed, loc):
    """Which quantity is solid and which is dashed in this panel."""
    handles = [Line2D([0], [0], color=solid[1], lw=curve_lw, label=solid[0]),
               Line2D([0], [0], color=dashed[1], lw=1.3, ls="--", alpha=0.7, label=dashed[0])]
    ax.legend(handles=handles, loc=loc, bbox_to_anchor=(1.0, 0.9), frameon=True, fontsize=11, handlelength=2.0)


alignment = [finite_mean(data, "prediction_right_delta", i) / proj0 for i in range(len(rhos))]
losses = [normalize_curve(finite_mean(data, "loss", i)) for i in range(len(rhos))]

fig = plt.figure(figsize=(figwidth, 0.55 * figwidth), layout="constrained")
outer = fig.add_gridspec(2, 2, height_ratios=[1.6, 1.0], width_ratios=[1.0, 1.0])
ax_t = fig.add_subplot(outer[0, 0])
ax_s = fig.add_subplot(outer[0, 1], sharey=ax_t)
ax_leg = fig.add_subplot(outer[1, 0])

snaps = outer[1, 1].subgridspec(1, 2, width_ratios=[0.1, 0.9])[0, 1].subgridspec(1, 3, wspace=0.0)
snap_axes = [fig.add_subplot(snaps[0, k]) for k in range(3)]
for ax in (ax_t, ax_s, *snap_axes):
    ax.grid(False)

# Left: timescale t. Alignment error decreases, loss frozen.
for i, rho in enumerate(rhos):
    ax_t.plot(times[mask], alignment[i][mask], color=rho_color(i), lw=curve_lw, zorder=2)
    ax_t.plot(times[mask], losses[i][mask], color=rho_color(i), **faint)
alignment_grid = np.linspace(0.0, 15.0, 300)
alignment_limit = theory.projector_curves(alignment_grid, rhos=rhos, include_zero_rho=True)["delta0_V"] / proj0
ax_t.plot(alignment_grid, alignment_limit, **limit_style)
ax_t.set_xlim(0.0, 10.0)
ax_t.set_ylim(-0.03, 1.05)
ax_t.set_xlabel(r"$t$")
ax_t.set_ylabel("normalized alignment error", color=ALIGN_COLOR)
ax_t.set_title(r"$t = \Theta(1)$: alignment, loss frozen", pad=16)

faint_arrow(ax_t, start=(2.1, 0.5), end=(6.0, 0.5), color=LOSS_COLOR)
quantity_legend(ax_t, ("alignment error", ALIGN_COLOR), ("loss", LOSS_COLOR), "upper right")

# Right: timescale s. Loss decays, alignment error already zero.
for i, rho in enumerate(rhos):
    s = times[mask] / np.log(1.0 / rho)
    ax_s.plot(s, losses[i][mask], color=rho_color(i), lw=curve_lw, zorder=2)
    ax_s.plot(s, alignment[i][mask], color=rho_color(i), **faint)
s_grid = np.linspace(0.0, 1.5, 300)
loss_limit = normalize_curve(theory.logtime_spectral_front(s_grid, x_grid)["loss_limit"])
ax_s.plot(s_grid, loss_limit, **limit_style)
ax_s.set_xlim(0.0, 1.1)
ax_s.set_xlabel(r"$s = t / \log(\rho^{-1})$")
ax_s.set_ylabel("normalized loss", color=LOSS_COLOR)
ax_s.tick_params(labelleft=True)
ax_s.set_title(r"$t \propto \log(\rho^{-1})$: loss decays", pad=16)
faint_arrow(ax_s, start=(0.33, 0.35), end=(0.07, 0.35), color=ALIGN_COLOR)
quantity_legend(ax_s, ("loss", LOSS_COLOR), ("alignment error", ALIGN_COLOR), "upper right")

# Snapshots of the singular values for the smallest rho, as a fraction of all singular values.
s_targets = np.array([0.35, 0.575, 0.8])
ind_rho = len(rhos) - 1
s_all = times / np.log(1.0 / rhos[ind_rho])
time_indices = np.array([int(np.argmin(np.abs(s_all - s))) for s in s_targets])
s_snap = s_all[time_indices]
sv = np.asarray(arr(data, "singular_values_z"), dtype=float)[ind_rho]
target = theory.kappa_star * theory.law.pdf(x_grid)

for k, s0 in enumerate(s_snap):
    ax_s.axvline(s0, color="black", lw=1.0, ls=":", alpha=0.8, zorder=0)
    ax_s.text(s0 - 0.03, 0.92, rf"$s_{k + 1}$", transform=ax_s.get_xaxis_transform(), ha="center", va="bottom", fontsize=12)

bins = np.linspace(0.25, 2.3, 36)
for k, (ax, it) in enumerate(zip(snap_axes, time_indices)):
    raw = np.asarray(sv[it], dtype=float).reshape(-1)
    raw = raw[np.isfinite(raw)]
    counts, edges = np.histogram(raw[raw >= bins[0]], bins=bins)
    widths = np.diff(edges)
    ax.bar(edges[:-1], counts / (raw.size * widths), width=widths, align="edge",
           color=COLORS["Z"], alpha=0.45, edgecolor="none")
    ax.plot(x_grid, target, color=COLORS["C"], ls="--", lw=1.5, alpha=0.75)
    ax.text(0.95, 0.93, rf"$s_{k + 1}$", transform=ax.transAxes, ha="right", va="top", fontsize=12)
    ax.set_xlim(0.25, 2.3)
    ax.set_ylim(0.0, 1.15 * float(np.max(target)))
    ax.set_yticks([])
    ax.tick_params(labelsize=9)
snap_axes[1].set_title("learning the spectrum")
snap_axes[1].set_xlabel("singular value", fontsize=11)

rho_handles = []
for i, rho in enumerate(rhos):
    exponent = int(round(np.log10(rho)))
    label = rf"$\rho=10^{{{exponent}}}$" if rho < 0.02 else rf"$\rho={rho}$"
    rho_handles.append(Line2D([0], [0], color=rho_color(i), lw=2.0, label=label))
rho_handles.append(Line2D([0], [0], color="black", lw=2.0, alpha=0.75, label=r"theory, $\rho\to0$"))
spectrum_handles = [
    Patch(color=COLORS["Z"], alpha=0.45,
          label="GD singular values,\n" + rf"$\rho=10^{{{int(round(np.log10(rhos[ind_rho])))}}}$"),
    Line2D([0], [0], color=COLORS["C"], ls="--", lw=1.5, alpha=0.75, label="target spectral \n" + "measure"),
]

ax_leg.axis("off")
x_sep = 0.58
legend_style = dict(frameon=False, fontsize=11, handlelength=1.7, labelspacing=0.8, borderaxespad=0.0)
rho_legend = ax_leg.legend(handles=rho_handles, ncol=2, loc="center right", bbox_to_anchor=(x_sep - 0.03, 0.56),
                           columnspacing=1.1, **legend_style)
ax_leg.add_artist(rho_legend)
spectrum_legend = ax_leg.legend(handles=spectrum_handles, ncol=1, loc="center left",
                                bbox_to_anchor=(x_sep + 0.03, 0.56), **legend_style)

fig.canvas.draw()
to_axes = ax_leg.transAxes.inverted()
box = Bbox.union([rho_legend.get_window_extent(), spectrum_legend.get_window_extent()]).transformed(to_axes)
pad_x, pad_y = 0.03, 0.08
ax_leg.add_patch(FancyBboxPatch(
    (box.x0 - pad_x, box.y0 - pad_y), box.width + 2 * pad_x, box.height + 2 * pad_y,
    boxstyle="round,pad=0,rounding_size=0.03", transform=ax_leg.transAxes,
    facecolor="white", edgecolor="0.8", lw=0.8, zorder=0, clip_on=False))
ax_leg.add_line(Line2D([x_sep, x_sep], [box.y0 - 0.5 * pad_y, box.y1 + 0.5 * pad_y], transform=ax_leg.transAxes,
                       color="0.8", lw=0.8, clip_on=False))

savefig(fig, "figure2_visual_abstract.pdf")
plt.show()

## Figure 3: Convergence of the prediction

Left panel: distance $d^{-1}\|Z(t)-Z_\infty\|_F^2$ to the limit, in both regimes. Right panels: histogram of the nonzero singular values of $Z$ at the last saved time, against $\kappa^*\phi_C$ and the threshold $x_*$.

In [ ]:
data_under = finite["fig3_under"]
data_over = finite["fig3_over"]

def plot_convergence(ax, data, label, color):
    times = finite_times(data)
    dist = finite_mean(data, "prediction_convergence_error_per_d", 0)
    mask = np.isfinite(times) & np.isfinite(dist) & (times > 0.0) & (dist > 0.0)
    ax.loglog(times[mask], dist[mask], color=color, lw=2.2, label=label)


def plot_prediction_infinity_hist(ax, data, theory, title, color):
    # Singular values of the simulated Z(t) at the last saved time (rho index 0).
    z_final = np.asarray(arr(data, "singular_values_z"), dtype=float)[0, -1]
    n_total = np.count_nonzero(np.isfinite(z_final))  # all singular values of all runs, zeros included
    vals = regular_values(z_final, remove_zero=True)
    grid = np.linspace(theory.law.support[0] - 1, theory.law.support[1] + 1, 500)
    dens = theory.kappa_star * np.asarray(theory.law.pdf(grid), dtype=float)
    bins = np.linspace(theory.law.support[0], theory.law.support[1], 38)
    if vals.size:
        counts, edges = np.histogram(vals, bins=bins)
        widths = np.diff(edges)
        heights = counts / (n_total * widths)
        ax.bar(edges[:-1], heights, width=widths, align="edge", color=color, alpha=0.30, edgecolor="none", label=rf"$\mu_t^Z$")
    ax.plot(grid, dens, color=COLORS["C"], lw=1.9, ls="--", label=r"$\kappa^*\phi_C$")
    if theory.paper_x_star > theory.law.support[0] + 1e-10:
        ax.axvline(theory.paper_x_star, color=COLORS["accent"], lw=1.2, ls="--", label=r"$x_*$")
    ax.set_xlim(theory.law.support[0], theory.law.support[1])
    ax.set_title(title)


theory_fig3_under = theory_from_config(data_under)
theory_fig3_over = theory_from_config(data_over)

ratio = 0.33

fig = plt.figure(figsize=(figwidth, ratio * figwidth), layout="constrained")
outer = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.0], wspace=0.05)
ax_conv = fig.add_subplot(outer[0, 0])
right = outer[0, 1].subgridspec(2, 1, hspace=0.)
ax_under = fig.add_subplot(right[0, 0])
ax_over = fig.add_subplot(right[1, 0], sharex=ax_under)

plot_convergence(ax_conv, data_under, "Underparameterized", COLORS["U"])
plot_convergence(ax_conv, data_over, "Overparameterized", COLORS["V"])
ax_conv.set_xlabel(r"time $t$") 
ax_conv.set_ylabel(r"$d^{-1}\|\mathbf Z(t) - \mathbf Z_\infty\|_F^2$")
ax_conv.legend(frameon=True, loc='lower left')
ax_conv.set_ylim(1e-10, 10)
ax_conv.set_xlim(0.1, 500)

plot_prediction_infinity_hist(ax_under, data_under, theory_fig3_under, r"Underparameterized", COLORS["U"])
plot_prediction_infinity_hist(ax_over, data_over, theory_fig3_over, r"Overparameterized", COLORS["V"])
ax_under.legend(frameon=True, loc=(0.73, 0.1))
ax_over.legend(frameon=True, loc=(0.73, 0.15))


ax_over.set_xlim(-0.1, 2.9)
ax_under.set_xlim(-0.1, 2.9)


savefig(fig, "figure3_prediction_spectra.pdf")
plt.show()

## Figure 4: Spectral density of the prediction

Histograms of the nonzero singular values of $Z(t)$ at five times, against the high-dimensional density $\mu_t^Z$, the density $\kappa^*\phi_C$ and the threshold $x_*$.

In [ ]:
data = finite["fig4"]

times = finite_times(data)
sv = np.asarray(arr(data, "singular_values_z"), dtype=float)[0]

theory = theory_from_config(data)
z_grid = np.linspace(0.01, law.support[1] * 1.2, 500)
positive = np.flatnonzero(times > 0)

ratio = 0.52
fig, axes_grid = plt.subplots(2, 3, figsize=(figwidth, ratio * figwidth), sharex=True, sharey=True, layout="constrained")
all_axes = np.asarray(axes_grid).ravel()
axes = all_axes[:len(positive)]
legend_axis = all_axes[-1]
legend_axis.axis("off")

density = theory.density_grid(z_grid, times, branches=("U",), eta=5e-3, include_prediction_from_u=True, verbose=True, progress_every=50)

threshold = theory.paper_x_star

for k, ax in enumerate(axes):
    vals = regular_values(sv[k], remove_zero=True)
    vals = vals[vals > z_grid[0]]
    if vals.size:
        lo = max(z_grid[0], np.nanmin(vals))
        hi = max(z_grid[-1], np.nanmax(vals))
        bins = np.linspace(lo, hi, 48)
        counts, edges = np.histogram(vals, bins=bins)
        widths = np.diff(edges)
        heights = counts / (sv[k].size * widths)  # all singular values of the 20 runs, zeros included
        ax.bar(edges[:-1], heights, width=widths, align="edge", color=COLORS["Z"], alpha=0.30, edgecolor="none", label=r"$\mu_t^Z$ (GD)")
    ax.plot(z_grid, density["density_Z_unnormalized"][k], color=COLORS["Z"], lw=2.4, label=r"$\mu_t^Z$ (theory)")
    ax.plot(z_grid, theory.kappa_star * law.pdf(z_grid), color=COLORS["C"], lw=1.6, ls="--", label=r"$\kappa^*\phi_C$")
    ax.axvline(threshold, color=COLORS["accent"], lw=1.2, ls="--", label=r"$x_*$")
    ax.set_xlim(z_grid[0], z_grid[-1])
    ax.set_ylim(-0.05, 1.3)
    ax.set_title(rf"$t={times[k]:.2g}$")
handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
order = [r"$\mu_t^Z$ (GD)", r"$\mu_t^Z$ (theory)", r"$\kappa^*\phi_C$", r"$x_*$"]
legend_labels = [r"GD singular values ($d=500$)", r"high-dimensional density $\mu_t^Z$",
                 r"density $\kappa^*\phi_C$", r"selection threshold $x_*$"]
legend_axis.legend([by_label[key] for key in order], legend_labels, frameon=False, loc="center left", bbox_to_anchor=(0.0, 0.53), handlelength=2.2)

axes[0].set_ylabel("density")
axes[3].set_ylabel("density")
for ax in axes[3:]:
    ax.set_xlabel(r"singular value $x$")

savefig(fig, "figure4_prediction_density.pdf")
plt.show()

## Figure 5: Range-projector alignment

Distances $\Delta^\mathsf{L}(t)^2$ and $\Delta^\mathsf{R}(t)^2$ of gradient descent against the finite-$\rho$ formulas of Theorem 6: underparameterized regime, and overparameterized regime with edge exponents $\ell = 0$ and $\ell = 1$. The grey lines are the asymptotic rates.

In [ ]:
ratio = 0.3
fig, axes = plt.subplots(1, 3, figsize=(figwidth, ratio * figwidth), sharey=False, layout="constrained")

from math import gamma

def figure5_theory_from_config(data, edge_ell=None):
    if edge_ell is None:
        return theory_from_config(data)
    cfg = loaded_dict(data)["config"]
    law_edge = SignalLaw.from_pdf(edge_power_pdf(edge_ell), (0.35, 2.0), normalize=False, name=f"edge_{edge_ell}")
    return make_theory(
        law_edge,
        kappa=float(cfg["kappa"]),
        tau=float(cfg["tau"]),
        kappa_star=float(cfg["kappa_star"]),
        rho=float(np.asarray(cfg["rhos"], dtype=float).reshape(-1)[0]),
        n_quad=600,
    )

def figure5_projector_asymptotic(theory, times, edge_ell=None):
    times = np.asarray(times, dtype=float)
    if theory.kappa < theory.signal_weight:
        const = theory.projector_asymptotic_constants()["under_t_delta_limit"]
        return const / times
    if edge_ell is None:
        return None
    ell = float(edge_ell)
    x0 = float(theory.law.support[0])
    width = float(theory.law.support[1] - theory.law.support[0])
    phi0 = (ell + 1.0) / (width ** (ell + 1.0))
    prefactor = 8.0 * theory.beta * theory.kappa_star * (1.0 - theory.kappa) / (theory.kappa - theory.signal_weight)
    const = prefactor * phi0 * gamma(ell + 1.0) / (2.0 ** (ell + 1.0))
    return const * (times ** (-(ell + 1.0))) * np.exp(-2.0 * x0 * times)

panels = [
    (finite["fig5_under"], r"$\kappa < \beta \kappa^*$", None),
    (finite["fig5_over_l0"], r"$\kappa > \beta \kappa^*$,   $\ell = 0$", 0),
    (finite["fig5_over_l1"], r"$\kappa > \beta \kappa^*$,   $\ell = 1$", 1),
]

for ax, (data, title, edge_ell) in zip(axes, panels):
    theory = figure5_theory_from_config(data, edge_ell=edge_ell)
    ft = finite_times(data)
    rhos = np.asarray(arr(data, "rhos"), dtype=float)
    positive_times = ft[ft > 0]
    tt = np.geomspace(max(positive_times[0], 1e-3), positive_times[-1], 420)

    curves = theory.projector_curves(tt, rhos=rhos, include_zero_rho=False)

    for i, rho in enumerate(rhos):
        color = rho_color(i)
        u_curve = np.asarray(curves["delta_U"][i], dtype=float)
        v_curve = np.asarray(curves["delta_V"][i], dtype=float)

        if has(data, "prediction_left_delta") and title == r"$\kappa < \beta \kappa^*$":
            vals_left = finite_mean(data, "prediction_left_delta", i)
            ax.plot(ft, vals_left, color=COLORS['U'], label=r"$\Delta^\mathsf{L}(t)^2$ (GD)")
        if has(data, "prediction_right_delta"):
            vals_v = finite_mean(data, "prediction_right_delta", i)
            ax.plot(ft, vals_v, color=COLORS['V'], label=r"$\Delta^\mathsf{R}(t)^2$ (GD)")

        if np.any(u_curve > 0):
            ax.plot(tt, u_curve, color=darken_color(COLORS['U']), lw=2.0, ls="dashed", label=r"$\Delta^\mathsf{L}(t)^2$ (Theory)")
        if np.any(v_curve > 0):
            ax.plot(tt, v_curve, color=darken_color(COLORS['V']), lw=2.0, ls="dashed", label=r"$\Delta^\mathsf{R}(t)^2$ (Theory)")

    asymptotic = figure5_projector_asymptotic(theory, tt, edge_ell=edge_ell)
    if asymptotic is not None:
        ax.plot(tt, asymptotic, color="0.5", lw=1.2, label="Asymptotic")
    
    ax.set_title(title)
    ax.set_xlabel(r"$t$")
    ax.set_yscale("log")
    ax.set_ylim(1e-8, 1.1)

axes[0].set_xscale('log')
axes[0].set_ylabel(r"$\Delta^{\mathsf{a}}(t)^2$")
axes[0].legend(frameon=True, fontsize=10, loc="lower left", ncol=1)
axes[1].legend(frameon=True, fontsize=10, loc="lower left", ncol=1)


axes[0].set_ylim(1e-3, 1)
axes[1].set_ylim(1e-40, 10)
axes[2].set_ylim(1e-40, 10)

axes[0].set_xlim(1e-2, 2e2)
axes[1].set_xlim(0., 110)
axes[2].set_xlim(0., 110)


plt.show()
savefig(fig, "figure5_range_projectors.pdf")

## Figure 6: Convergence rates of the top singular spaces

Left: gapped spectrum, the distance decays like $e^{-2\epsilon_\alpha t}$ (semilog axes). Right: continuous spectrum at the cutoff, the distance decays like $t^{-1}$ (log-log axes). Bands span the left and right distances.

In [ ]:
from matplotlib.patches import FancyBboxPatch

continuous_data = finite["fig6_continuous"]
gapped_data = finite["fig6_gapped"]

continuous_payload = loaded_dict(continuous_data)
gapped_payload = loaded_dict(gapped_data)

ratio = 0.35
fig, axes = plt.subplots(1, 2, figsize=(figwidth, ratio * figwidth), layout="constrained")

def top_projector_scale(payload):
    """Factor k/d converting the stored distances to the normalization of Theorem 10.

    The simulator (src/simulator.py) and the rho -> 0 envelope (figures/figure06.py) divide
    ||P_k - P_k^C||_F^2 by k; the theorem divides by d.
    """
    d = int(payload["dimensions"]["d"])
    frac = float(np.asarray(payload["top_eig_fracs"], dtype=float).reshape(-1)[0])
    k = int(np.floor(frac * d))
    return k / d


def projector_rate_curves(payload):
    """Left/right top-k distances of the singular projectors of Z = U V^T, normalized by d as in Theorem 10."""
    left_key, right_key = "prediction_top_left_delta", "prediction_top_right_delta"
    scale = top_projector_scale(payload)
    left = scale * np.mean(np.asarray(payload[left_key], dtype=float), axis=-1)[:, :, 0]
    right = scale * np.mean(np.asarray(payload[right_key], dtype=float), axis=-1)[:, :, 0]
    rho0_left = scale * np.mean(np.asarray(payload["rho0_prediction_top_left_delta"], dtype=float), axis=-1)
    rho0_right = scale * np.mean(np.asarray(payload["rho0_prediction_top_right_delta"], dtype=float), axis=-1)
    return left, right, rho0_left, rho0_right, scale

for ax, payload, regime in (
    (axes[0], gapped_payload, "gapped"),
    (axes[1], continuous_payload, "continuous"),
):
    times = np.asarray(payload["times"], dtype=float)
    rhos = np.asarray(payload["rhos"], dtype=float)
    left, right, rho0_left, rho0_right, scale = projector_rate_curves(payload)
    for rho_index, rho in enumerate(rhos):
        color = rho_color(rho_index)
        center = 0.5 * (left[rho_index] + right[rho_index])
        lower = np.minimum(left[rho_index], right[rho_index])
        upper = np.maximum(left[rho_index], right[rho_index])
        valid = (times > 0.0) & np.isfinite(center) & (center > 1e-14)
        ax.fill_between(times[valid], lower[valid], upper[valid], color=color, alpha=0.12, linewidth=0)
        exponent = int(round(np.log10(rho)))
        ax.plot(times[valid], center[valid], color=color, lw=1.25, alpha=0.90, label=rf"$\rho=10^{{{exponent}}}$")

    envelope = 0.5 * (rho0_left + rho0_right)
    envelope_lower = np.minimum(rho0_left, rho0_right)
    envelope_upper = np.maximum(rho0_left, rho0_right)
    envelope_valid = (times > 0.0) & np.isfinite(envelope) & (envelope > 1e-14)
    ax.fill_between(times[envelope_valid], envelope_lower[envelope_valid], envelope_upper[envelope_valid], color=COLORS["C"], alpha=0.13, linewidth=0)
    ax.plot(times[envelope_valid], envelope[envelope_valid], color=COLORS["C"], lw=1.5, ls='--', zorder=5, label="Envelope")

    density_grid = np.asarray(payload["teacher_density_grid"], dtype=float)
    density = np.asarray(payload["teacher_density"], dtype=float)
    x_alpha = float(np.asarray(payload["x_alpha"]))
    density_ax = ax.inset_axes([0.61, 0.66, 0.32, 0.25])

    density_ax.plot(density_grid, density, color=COLORS["Z"], lw=1.25)
    density_ax.axvline(x_alpha, color=COLORS["accent"], lw=1.1)
    density_ax.set_xlim(0.7, 1.8)


    background = FancyBboxPatch((-0.1, -0.3), 1.2, 1.55, transform=density_ax.transAxes, boxstyle="round,pad=0.02", 
                                facecolor="white", edgecolor="0.3", linewidth=1, zorder=-10, clip_on=False)
    density_ax.add_patch(background)
    
    density_ax.set_yticks([])
    density_ax.tick_params(axis="x", labelsize=6.5, length=2)

    if regime == "continuous":
        reference_times = np.geomspace(1.0, 35.0, 80)
        anchor_time = 8.0
        anchor = np.interp(anchor_time, times[envelope_valid], envelope[envelope_valid])
        reference = anchor * (reference_times / anchor_time) ** (-1.0)
        ax.plot(reference_times, 2 * reference, color=COLORS["muted"], lw=1.35, ls="--")
        ax.text(2.5, 3.4 * np.interp(2.5, reference_times, reference), r"$\propto t^{-1}$", color=COLORS["muted"], fontsize=12, va="top")
        ax.set_xscale("log")
        ax.set_xlim(0.7, 40.0)
        ax.set_ylim(1e-3 * scale, 1.5 * scale)
        ax.set_title(r"Continuous spectrum")
        density_ax.set_ylim(0.2, 1.0)
        density_ax.text(x_alpha, density_ax.get_ylim()[1] + 0.05, r"$x_\alpha$", ha="center", va="bottom", fontsize=12, color=COLORS["accent"])

    else:
        epsilon_alpha = float(np.asarray(payload["epsilon_alpha"]))
        reference_times = np.linspace(10.0, 80.0, 100)
        anchor_time = 15.0
        anchor = np.interp(anchor_time, times[envelope_valid], envelope[envelope_valid])
        reference = anchor * np.exp(-2.0 * epsilon_alpha * (reference_times - anchor_time))
        ax.plot(reference_times, 5 * reference, color=COLORS["muted"], lw=1.35, ls="--", label="Asymptotic")
        ax.text(14, 12 * np.interp(10.5, reference_times, reference), r"$\propto e^{-2\epsilon_\alpha t}$", color=COLORS["muted"], fontsize=12, va="top")
        ax.set_xlim(0.5, 40.0)
        ax.set_ylim(1e-12 * scale, 1.5 * scale)
        ax.set_title(r"Gapped spectrum")
        density_ax.axvspan(x_alpha - epsilon_alpha, x_alpha, color=COLORS["accent"], alpha=0.18, linewidth=0)
        density_ax.set_ylim(-0.2, 1.5)
        density_ax.text(x_alpha, density_ax.get_ylim()[1] + 0.05, r"$x_\alpha$", ha="center", va="bottom", fontsize=12, color=COLORS["accent"])


    ax.set_yscale("log")
    ax.set_xlabel(r"$t$")

handles, labels = axes[0].get_legend_handles_labels()
axes[1].legend(handles, labels, loc=(1.0, 0.1))
axes[0].set_ylabel(r"$\Delta_{\alpha,\rho}^2(t)$")

savefig(fig, "figure6_top_eigenspaces.pdf")
plt.show()

## Figure 7: Saddle-to-saddle dynamics

Normalized loss $d^{-1}\lVert Z(t)-C\rVert_F^2$ against $s = t/\log(1/\rho)$, for decreasing $\rho$. At $d=24$ the loss goes down in steps, one per singular value; at $d=96$ the steps are denser and close to the high-dimensional curve.

In [ ]:
small_data = finite["fig7_rho"]
large_data = finite["fig7_large"]

small_payload = loaded_dict(small_data)
large_payload = loaded_dict(large_data)

ratio = 0.32
fig, axes = plt.subplots(1, 2, figsize=(figwidth, ratio * figwidth), sharey=True, layout="constrained")

s_reference = np.linspace(0.20, 0.68, 2401)

def plot_loss_family(ax, payload):
    times = np.asarray(payload["times"], dtype=float)
    rhos = np.asarray(payload["rhos"], dtype=float)
    losses = np.mean(np.asarray(payload["loss_per_d"], dtype=float), axis=2)
    for i, rho in enumerate(rhos):
        s_rho = times / np.log(1.0 / rho)
        rho_exponent = int(round(np.log10(rho)))
        ax.plot(s_rho, losses[i], color=rho_color(i), lw=1.55, label=rf"$\rho=10^{{{rho_exponent}}}$")

    singular_values = np.asarray(payload["teacher_singular_values"], dtype=float)[0]
    selected_rank = int(payload["dimensions"]["selected_rank"])
    d = int(payload["dimensions"]["d"])
    learned_singular_values = singular_values[:selected_rank]
    acquired_energy = np.sum(learned_singular_values[:, None] ** 2 * (s_reference[None, :] >= 1.0 / (2.0 * learned_singular_values[:, None])), axis=0)
    exact_loss = (np.sum(singular_values ** 2) - acquired_energy) / d
    ax.plot(s_reference, exact_loss, color=COLORS["C"], lw=1.4, drawstyle="steps-post", label=r"$\rho \to 0$ (theory)")

plot_loss_family(axes[0], small_payload)
plot_loss_family(axes[1], large_payload)

for ax in axes:
    ax.set_xlim(0.20, 0.68)
    ax.set_xlabel(r"$s=t/\log(1/\rho)$")
axes[0].set_ylabel(r"$d^{-1} \|\mathbf{Z}(t) - \mathbf{C}\|_F^2$")
axes[0].set_title(r"$d = 24$")
axes[1].set_title(r"$d = 96$")
axes[0].legend(frameon=True, loc="upper right")

savefig(fig, "figure7_spectral_staircase.pdf")
plt.show()

## Figure 8: Spectral measures at logarithmic times

Nonzero singular values of $Z(t)$ at fixed values of $s = t/\log(1/\rho)$, against the limiting density, with the mass below the moving threshold $x_s$ and the loss.

In [ ]:
data = finite["fig8"]
s_targets = np.array([0.31, 0.36, 0.41, 0.46])
rho = float(np.asarray(arr(data, "rhos"), dtype=float)[0])
times = finite_times(data)
theory = theory_from_config(data)
x_grid_fig8 = np.linspace(0.01, 3.0, 500)
x_star = theory.paper_x_star

log_rho = np.log(1.0 / rho)
s_all = times / log_rho
sv_rho = np.asarray(arr(data, "singular_values_z"), dtype=float)[0]
time_indices_rho = np.array([int(np.argmin(np.abs(s_all - s))) for s in s_targets])
density_times_rho = times[time_indices_rho]
s_true_rho = s_all[time_indices_rho]
front_rho = theory.logtime_spectral_front(s_true_rho, x_grid_fig8)
front_all = theory.logtime_spectral_front(s_all, x_grid_fig8)

# Accurate finite-rho density at the four selected times.
z_grid_rho = np.linspace(0.25, 3.0, 320)
theory_rho = make_theory(
    theory.law, kappa=theory.kappa, tau=theory.tau,
    kappa_star=theory.kappa_star, rho=rho, n_quad=2000,
)
density_rho = theory_rho.density_grid(
    z_grid_rho, density_times_rho, branches=("U",), eta=5e-3,
    include_prediction_from_u=True, verbose=True,
)


ratio = 0.48

fig = plt.figure(figsize=(figwidth, ratio * figwidth), layout="constrained")

outer = fig.add_gridspec(3, 1, height_ratios=(1.0, 0.18, 0.75))

density_grid = outer[0].subgridspec(1, len(s_targets), wspace=0.06)
bottom_grid = outer[2].subgridspec(1, 2, wspace=0.03)


density_axes = []
for ell in range(len(s_targets)):
    density_axes.append(fig.add_subplot(density_grid[0, ell], sharey=density_axes[0] if density_axes else None))
mass_ax = fig.add_subplot(bottom_grid[0, 0])
loss_ax = fig.add_subplot(bottom_grid[0, 1], sharex=mass_ax)

for ell, (ax, s, dens, threshold) in enumerate(zip(
    density_axes, s_true_rho, front_rho["density_Z"], front_rho["threshold"],
)):
    effective_threshold = max(float(threshold), float(theory.paper_x_star), float(theory.law.support[0]))
    raw_vals = np.asarray(sv_rho[int(time_indices_rho[ell])], dtype=float).reshape(-1)
    raw_vals = raw_vals[np.isfinite(raw_vals)]
    vals = regular_values(raw_vals, remove_zero=True, zero_tol=1e-6)
    bins = np.linspace(0.25, 3.0, 80)
    vals_hist = vals[vals >= bins[0]]
    counts, edges = np.histogram(vals_hist, bins=bins)
    widths = np.diff(edges)
    heights = counts / (max(raw_vals.size, 1) * widths)

    ax.bar(edges[:-1], heights, width=widths, align="edge", color=COLORS["Z"], alpha=0.28, edgecolor="none", label=r"$\mu_t^Z$ (GD)")
    ax.plot(x_grid_fig8, dens, color=COLORS["Z"], lw=2.4, label=r"$\mu_t^Z$ (theory, $\rho\to0$)")
    ax.plot(z_grid_rho, density_rho["density_Z_unnormalized"][ell], color="0.45", lw=1.7, label=r"$\mu_t^Z$ (theory, finite $\rho$)")
    ax.plot(x_grid_fig8, theory.kappa_star * theory.law.pdf(x_grid_fig8), color=COLORS["C"], lw=1.6, ls="--", label=r"$\kappa^*\phi_C$")
    ax.axvline(effective_threshold, color=COLORS["accent"], lw=1.5, ls=":", label=r"$x_s$")
    ax.axvline(x_star, color="black", lw=1.5, ls=":", label=r"$x_*$")
    ax.set_title(rf"$s_{ell + 1}={s:.2g}$")
    ax.set_xlim(0.25, 2.3)
    ax.set_ylim(0.0, 1.2)
    ax.set_xlabel(r"$x$", labelpad=-5)
    if ell > 0:
        ax.tick_params(labelleft=False)

density_axes[0].set_ylabel(r"Density on $x>0$")
density_handles, density_labels = density_axes[0].get_legend_handles_labels()

order = [0, 1, 5, 2, 3, 4]
density_handles = [density_handles[o] for o in order]
density_labels = [density_labels[o] for o in order]

fig.legend(density_handles, density_labels, loc="center", bbox_to_anchor=(0.53, 0.44), ncol=6, frameon=True, columnspacing=1.15, handlelength=1.8)

# Empirical mass below the moving threshold, using every stored GD time.
threshold_floor = max(float(theory.paper_x_star), float(theory.law.support[0]))
effective_threshold_all = np.maximum(np.asarray(front_all["threshold"], dtype=float), threshold_floor)
sv_flat = sv_rho.reshape(len(times), -1)
finite_mask = np.isfinite(sv_flat)
mass_below_xs = np.sum(finite_mask & (np.abs(sv_flat) < effective_threshold_all[:, None]), axis=1) / np.maximum(np.sum(finite_mask, axis=1), 1)
theory_atom = np.asarray(front_all["atom_zero"], dtype=float)
permanent_atom = max(0.0, 1.0 - theory.selected_mass / theory.beta)

mass_ax.fill_between(s_all, 0.0, permanent_atom, color="0.82", label=r"Permanent atom")
mass_ax.fill_between(s_all, permanent_atom, theory_atom, color=COLORS["Z"], alpha=0.18, label=r"Waiting atom")
mass_ax.plot(s_all, theory_atom, color=COLORS["Z"], lw=2.0, label=r"Atom at zero")
mass_ax.plot(s_all, mass_below_xs, color=COLORS["accent"], lw=1.6, label=r"Mass below $x_s$ (GD)")
mass_ax.scatter(s_true_rho, mass_below_xs[time_indices_rho], color=COLORS["accent"], edgecolor="white", linewidth=0.7, s=32, zorder=6)
mass_ax.set_ylim(0.0, 1.02)
mass_ax.set_ylabel(r"Spectral mass")
mass_ax.set_xlabel(r"$s=t/\log(1/\rho)$")
mass_ax.set_title(r"Mass at zero")
mass_ax.legend(frameon=True, loc=(1.02, 0.1), fontsize=11)
mass_ax.set_xlim(0.1, 0.7)
mass_ax.set_ylim(0.0, 1.1)


# Loss along the same log-time trajectory.
# Divided by min(p, d), like loss_limit (Theorem 13).
loss_runs = np.asarray(arr(data, "loss"), dtype=float)[0]
loss_mean = np.nanmean(loss_runs, axis=-1)
loss_std = np.nanstd(loss_runs, axis=-1)
loss_limit = np.asarray(front_all["loss_limit"], dtype=float)
loss_ax.fill_between(s_all, np.maximum(loss_mean - loss_std, 0.0), loss_mean + loss_std, color=COLORS["accent"], alpha=0.12, linewidth=0)
loss_ax.plot(s_all, loss_mean, color=COLORS["accent"], lw=1.8, label=r"GD")
loss_ax.plot(s_all, loss_limit, color=COLORS["C"], lw=1.8, ls="--", label=r"$\rho\to0$ theory")
loss_ax.scatter(s_true_rho, loss_mean[time_indices_rho], color=COLORS["accent"], edgecolor="white", linewidth=0.7, s=32, zorder=6)
loss_ax.set_ylabel(r"$d^{-1}\|\mathbf{Z}(t)-\mathbf{C}\|_F^2$")
loss_ax.set_xlabel(r"$s=t/\log(1/\rho)$")
loss_ax.set_title(r"Loss")
loss_ax.set_xlim(0.1, 0.7)
loss_ax.legend(frameon=True, loc="upper right")

for ell, s in enumerate(s_true_rho):
    for ax in (mass_ax, loss_ax):
        ax.axvline(s, color='black', lw=1.0, ls=":", alpha=0.8, zorder=0)

savefig(fig, "figure8_spectral_measures.pdf")
plt.show()